# QC filtering on a UMAP

For each sample, the raw cells are put on a UMAP and coloured by whether the QC cutoffs would remove them, once for the
default fixed cutoffs (15 counts, 10 genes), once for the Gaussian-mixture cutoffs and once for the sparse-tissue enrichment cutoffs (same rules as `02_qc_filtering.ipynb`).
The UMAP is a throwaway diagnostic computed in memory on a copy of each raw sample; no objects are written, only figures.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
import lamindb as ln
from matplotlib.lines import Line2D
from scipy.stats import binom, poisson
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

In [ ]:
SAMPLE_NAMES = [
    'TIS04399-001',
    'TIS02434-001',
    'TIS02437-001',
    'TIS04398-001',
    'TIS02611-001',
    'TIS02603-001',
    'TIS02604-001',
    'TIS02613-001',
    'TIS02421-001',
    'TIS02430-001',
    'TIS02424-001',
    'TIS02439-001',
    'TIS02594-001',
]

OUTS_DIR    = '/home/workspace/PCD_data/xenium/fh1_pretreatment/data/outs'
FIGURES_DIR = '/home/workspace/PCD_data/xenium/fh1_pretreatment/figures/qc_filtering_umap'

PREVIOUS_MIN_COUNTS = 15
PREVIOUS_MIN_GENES  = 10

NOISE_QUANTILE = 0.999

NEIGHBORS          = 10
SPARSE_FRACTION    = 0.10
STEP_WIDTH         = 5
MIN_CELLS_PER_STEP = 30
ALPHA              = 0.05
MAX_COUNTS         = 60

os.makedirs(FIGURES_DIR, exist_ok=True)

## Cutoffs

Same rules as the QC notebook: noise floor, Gaussian mixture (counts and genes), and the enrichment cutoff on counts.
The enrichment rule only sets a counts cutoff, so it is combined with the mixture genes cutoff.
A sample with no quiet step has an enrichment cutoff of NaN and removes nothing by counts.

In [ ]:
samples = {}
cutoffs = pd.DataFrame(index=SAMPLE_NAMES, dtype=float)
for name in SAMPLE_NAMES:
    # Loads the raw per-sample AnnData from lamindb storage.
    adata = ln.Artifact.get(key=f'{name}.h5ad').load()
    # Adds obs['n_genes_by_counts'] and overwrites obs['total_counts']; .X is not changed; in memory only.
    sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)
    samples[name] = adata

    counts = adata.obs['total_counts'].to_numpy()
    genes  = adata.obs['n_genes_by_counts'].to_numpy()

    false_positives = float(pd.read_csv(f'{OUTS_DIR}/{name}/metrics_summary.csv').iloc[0]['estimated_number_of_false_positive_transcripts_per_cell'])
    noise_min_counts = poisson.ppf(NOISE_QUANTILE, false_positives) + 1

    mixture_cuts = {}
    for metric, values in [('counts', counts), ('genes', genes)]:
        log_values     = np.log10(values[values > 0]).reshape(-1, 1)
        two_components = GaussianMixture(2, random_state=0).fit(log_values)
        low  = np.argmin(two_components.means_.ravel())
        grid = np.linspace(two_components.means_.min(), two_components.means_.max(), 2000).reshape(-1, 1)
        posterior_low = two_components.predict_proba(grid)[:, low]
        mixture_cuts[metric] = np.ceil(10 ** grid[np.argmax(posterior_low < 0.5), 0])

    coordinates = adata.obsm['spatial']
    neighbor_distance = NearestNeighbors(n_neighbors=NEIGHBORS + 1).fit(coordinates).kneighbors(coordinates)[0][:, -1]
    in_sparse_tissue  = neighbor_distance > np.quantile(neighbor_distance, 1 - SPARSE_FRACTION)

    first_quiet_step_start = np.nan
    for step_start in range(0, MAX_COUNTS, STEP_WIDTH):
        in_step        = (counts >= step_start) & (counts < step_start + STEP_WIDTH)
        cells_in_step  = in_step.sum()
        lonely_in_step = (in_step & in_sparse_tissue).sum()
        p_value = binom.sf(lonely_in_step - 1, cells_in_step, SPARSE_FRACTION)
        if cells_in_step >= MIN_CELLS_PER_STEP and p_value >= ALPHA:
            first_quiet_step_start = step_start
            break

    cutoffs.loc[name, 'min_genes']             = mixture_cuts['genes']
    cutoffs.loc[name, 'mixture_min_counts']    = max(noise_min_counts, mixture_cuts['counts'])
    cutoffs.loc[name, 'enrichment_min_counts'] = max(noise_min_counts, first_quiet_step_start)

    # Adds obs['removed_default'], obs['removed_mixture'] and obs['removed_enrichment'] (bool, True = removed); in memory only.
    adata.obs['removed_default']    = (counts < PREVIOUS_MIN_COUNTS) | (genes < PREVIOUS_MIN_GENES)
    adata.obs['removed_mixture']    = (counts < cutoffs.loc[name, 'mixture_min_counts']) | (genes < cutoffs.loc[name, 'min_genes'])
    adata.obs['removed_enrichment'] = (counts < cutoffs.loc[name, 'enrichment_min_counts']) | (genes < cutoffs.loc[name, 'min_genes'])

cutoffs

## UMAPs of the raw cells

The UMAP is computed on a copy of each sample (normalize, log1p, PCA, neighbours, UMAP) so the raw samples above stay unchanged.
Both panels of a row share the same coordinates, so the only difference is which cells are red.

In [ ]:
umaps = {}
for name, adata in samples.items():
    umap_input = adata.copy()
    # On the copy only: normalize_total and log1p overwrite .X; PCA, neighbours and UMAP add obsm['X_pca'], obsp and obsm['X_umap'].
    sc.pp.normalize_total(umap_input)
    sc.pp.log1p(umap_input)
    sc.pp.pca(umap_input)
    sc.pp.neighbors(umap_input)
    sc.tl.umap(umap_input)
    umaps[name] = umap_input.obsm['X_umap']

In [ ]:
RULES = [('removed_default', 'default'), ('removed_mixture', 'mixture'), ('removed_enrichment', 'enrichment')]

fig, axes = plt.subplots(len(samples), len(RULES), figsize=(6 * len(RULES), 5.5 * len(samples)), squeeze=False)

for i, (name, adata) in enumerate(samples.items()):
    for j, (column, rule_name) in enumerate(RULES):
        ax      = axes[i, j]
        removed = adata.obs[column].to_numpy()
        ax.scatter(umaps[name][~removed, 0], umaps[name][~removed, 1], s=1, c='lightgrey', rasterized=True)
        ax.scatter(umaps[name][removed, 0], umaps[name][removed, 1], s=1, c='red', rasterized=True)
        ax.set_title(f'{name} — {rule_name}\n{removed.sum():,} of {len(removed):,} removed ({removed.mean():.1%})')
        ax.set_xlabel('UMAP1')
        ax.set_ylabel('UMAP2')
        ax.legend(
            handles=[
                Line2D([], [], marker='o', ls='', color='lightgrey', label='kept'),
                Line2D([], [], marker='o', ls='', color='red', label='removed'),
            ],
            loc='best',
        )

fig.tight_layout()
# for extension in ['png', 'pdf']:
#     # Writes the figure to disk in the new qc_filtering_umap folder; no data objects are touched.
#     fig.savefig(os.path.join(FIGURES_DIR, f'umap_removed_cells.{extension}'), dpi=150, bbox_inches='tight')
plt.show()

## Per-cell QC histograms

Total transcripts, unique transcripts, cell area and nucleus ratio for each raw sample. Each sample has three rows, one per rule (default, mixture, enrichment);
cells the rule removes are red and stacked on top of the kept cells in grey.

In [ ]:
fig, axes = plt.subplots(len(RULES) * len(samples), 4, figsize=(15, 3.5 * len(RULES) * len(samples)), squeeze=False)

for i, (name, adata) in enumerate(samples.items()):
    plot_data = pd.DataFrame({
        'Total transcripts per cell': adata.obs['total_counts'],
        'Unique transcripts per cell': adata.obs['n_genes_by_counts'],
        'Area of segmented cells': adata.obs['cell_area'],
        'Nucleus ratio': adata.obs['nucleus_area'] / adata.obs['cell_area'],
    })
    for k, (column, rule_name) in enumerate(RULES):
        plot_data['removed'] = adata.obs[column]
        for j, metric in enumerate(plot_data.columns[:4]):
            ax = axes[len(RULES) * i + k, j]
            sns.histplot(
                data=plot_data, x=metric, hue='removed', hue_order=[False, True],
                palette={False: 'lightgrey', True: 'red'}, multiple='stack',
                kde=False, ax=ax, legend=False,
            )
            ax.set_title(f'{name} — {rule_name}\n{metric}' if j == 0 else metric)

fig.tight_layout()
# for extension in ['png', 'pdf']:
#     # Writes the figure to disk in the qc_filtering_umap folder under a new stem; no data objects are touched.
#     fig.savefig(os.path.join(FIGURES_DIR, f'qc_histograms_by_removal.{extension}'), dpi=150, bbox_inches='tight')
plt.show()